<a href="https://colab.research.google.com/github/claudiohenriquezberroeta-pucv/desafio_kaggle_5/blob/versi%C3%B3n_3/Desafio_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
import kagglehub                #para importar datos del desafio
from google.colab import files  #para guardar datos en Drive (características y tokens)

from sklearn.cluster import KMeans    #para la clusterización en la función de tokenización
from sklearn.metrics import f1_score  #para utilizar la métrica de rendimiento f1_score
from sklearn.model_selection import train_test_split    #para la división de datos en conjuntos de entrenamiento y test

import pandas as pd             #para la manipulación de datos
import numpy as np              #para la manipulación de arreglos
import os                       #para operar con los archivos y directorios
import librosa                  #para el procesamiento de señales de audio
import librosa.display          #para visualizar de forma gráfica características de audio
import tensorflow as tf         #para trabajar con redes y tensores

#Elementos de tensorflow
from tensorflow.keras.utils import to_categorical       #para la converción de datos categórico (one hot)
from tensorflow.keras.layers import Input               #para definir la forma de los datos de entrada
from tensorflow.keras.layers import Dense               #para crear y definir la capa densa
from tensorflow.keras.layers import Dropout             #para definir la técnica de regularización dropout
from tensorflow.keras.layers import Embedding           #para la conversión de categorías en vectores densos de tamaño fijo
from tensorflow.keras.layers import GlobalMaxPooling1D  #para realizar la reducción (pooling) en datos temporales o secuenciales
from tensorflow.keras.models import Sequential          #para inicializar una pila lineal de capas
from tensorflow.keras.callbacks import EarlyStopping    #para definir criterios de detección temprana
from tensorflow.keras import regularizers               #para definir regulizadores L1 L2
from tensorflow.keras.preprocessing.sequence import pad_sequences   #para normalizar la longitud de listas de secuencias, transformándolas en un arreglo rectangular uniforme


In [ ]:
# 1. Subir el archivo kaggle.json
files.upload()

# 2. Crear la carpeta oculta .kaggle y mover el archivo allí
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# 3. Cambiar los permisos para que el archivo sea seguro
!chmod 600 ~/.kaggle/kaggle.json

# 4. Verificar la conexión listando datasets populares
!kaggle datasets list

Saving kaggle.json to kaggle.json
ref                                                              title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
algozee/teenager-menthal-healy                                   Social Media Impact on Teen Mental Health               16190  2026-04-05 08:04:21.823000          26463        562                1  
sridipbasu/ai-depndency-career-anxiety-and-student-burnout       AI Depndency, Career Anxiety and Student Burnout       447453  2026-05-10 18:26:03.943000            578         26                1  
sharmajicoder/gen-z-social-media-usage-dataset                   Gen-Z Social Media Usage Dataset                     44185801  2026-04-25 08:23:33.093000           3

In [ ]:
path = kagglehub.competition_download('clasificacion-de-generos-musicales')
print("Path to competition files:", path)

100%|██████████| 6.34G/6.34G [01:04<00:00, 106MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/clasificacion-de-generos-musicales


## Conexión con colab para guardar procesamiento

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Funciones de extracciones de características**

#Características espectrograma
*Función para extraer las características de un archivo de audio*

In [4]:
def extraer_espectrograma(archivo_mp3):
    y, sr = librosa.load(archivo_mp3, duration=30)

    # Generar el Espectrograma de Mel
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)

    # Convertir a decibelios (escala logarítmica)
    log_S = librosa.power_to_db(S, ref=np.max)

    return log_S

#Características tradicionales
*Función para extraer las características de un archivo de audio*

In [5]:
def obtener_caracteristicas_tradicionales(ruta_archivo):
    y, sr = librosa.load(ruta_archivo)

    # Transformar al dominio de la frecuencia y calcular MFCCs [cite: 151, 152]
    #n_mfcc=13 es el estándar para capturar el espectro de potencia [cite: 219]
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    return mfccs.T

# Tokenización - Vector Quantización VQ
*Función para tokenizar las características de los audios, devuelve los tokens y los centroides*

In [6]:
# Simulando un Codebook con K-Means (como hace HuBERT) [cite: 245, 265]
def tokenizar_con_vq(caracteristicas_audio, n_tokens=100):
    # caracteristicas_audio debe ser un array de forma (n_muestras, n_features)

    # Entrenar el "Codebook" (Acoustic Unit Discovery System) [cite: 245]
    kmeans = KMeans(n_clusters=n_tokens, random_state=42)
    kmeans.fit(caracteristicas_audio)

    # Asignar cada segmento de audio al token (índice del centroide) más cercano [cite: 69, 144]
    tokens_discretos = kmeans.predict(caracteristicas_audio)

    return tokens_discretos, kmeans.cluster_centers_

#**Extracción de características y tokenización de señales de audio**
Seteo de rutas y creación de listas

In [7]:
#Seteo de rutas y listas vacías
ruta = '/root/.cache/kagglehub/competitions/clasificacion-de-generos-musicales/train'
ruta_salida = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Train/'
ruta_test = '/root/.cache/kagglehub/competitions/clasificacion-de-generos-musicales/test'
ruta_test_salida = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Test/'

espectrograma = []
mfccs = []
caracteristicas = []
tokens_mfccs = []
tokens_espectrograma = []
tokens = []
codebook_mfccs = []
codebook_espectrograma = []
codebook =[]
errores = []

#Procesamiento de datos de señales de audio

##Datos de Entrenamiento

In [ ]:
#Ejecutar solo en caso de querer volver a procesar las señales de audio

for raiz, directorios, archivos in os.walk(ruta):
    for archivo in archivos:
        ruta_completa = os.path.join(raiz, archivo)
        try:
          #Obtención de características utilizando las funciones creadas
          feature_espectograma, sr = extraer_espectrograma(ruta_completa)
          feature_mfccs = obtener_caracteristicas_tradicionales(ruta_completa)
          #graficos de espectrogramas
          #graficar_espectrograma(feature_espectograma,sr)
        except:
          print('Error, no fue posible extraer caracteristicas del archivo',archivo)
          errores.append(archivo)
          with open(ruta_salida+'errores.txt', 'w', newline='', encoding='utf-8') as file:
            file.write(archivo)
        try:
          #guardando caracaterísticas en listas
          espectrograma.append(feature_espectograma)
          mfccs.append(feature_mfccs)
        except:
          print('Error, no fue posible guardar las caracteristicas en el listado',archivo)
          errores.append(archivo)
          with open(ruta_salida+'errores.txt', 'w', newline='', encoding='utf-8') as file:
            file.write(archivo)
        try:
          # generando tokens asociados a características
          mis_tokens_espectrograma, mi_codebook_espectrograma = tokenizar_con_vq(feature_espectograma)
          mis_tokens_mfccs, mi_codebook_mfccs = tokenizar_con_vq(feature_mfccs)

          # guardando tokens
          tokens_mfccs.append(mis_tokens_mfccs)
          tokens_espectrograma.append(mis_tokens_espectrograma)

          # guardando codebooks
          codebook_mfccs.append(mi_codebook_mfccs)
          codebook_espectrograma.append(mi_codebook_espectrograma)
        except:
          print('Error, no fue posible tokenizar las caracteristicas del archivo',archivo)
          errores.append(archivo)
          with open(ruta_salida+'errores.txt', 'w', newline='', encoding='utf-8') as file:
            file.write(archivo)

        np.save(ruta_test_salida+'espectrogramas/'+archivo[:-4]+'_espectrograma_test.npy', feature_espectograma)
        np.save(ruta_test_salida+'mfccs/'+archivo[:-4]+'_mfccs_test.npy', feature_mfccs)
        np.save(ruta_test_salida+'tokens_mfccs/'+archivo[:-4]+'_tokens_mfccs_test.npy', mis_tokens_mfccs)
        np.save(ruta_test_salida+'tokens_espectro/'+archivo[:-4]+'_tokens_espectrograma_test.npy', mis_tokens_espectrograma)
        np.save(ruta_test_salida+'codebooks_mffc/'+archivo[:-4]+'_codebook_mfccs_test.npy', mi_codebook_mfccs)
        np.save(ruta_test_salida+'codebooks_espectro/'+archivo[:-4]+'_codebook_espectrograma_test.npy', mi_codebook_espectrograma)

##Datos de Test

In [ ]:
#Ejecutar solo en caso de querer volver a procesar las señales de audio

for raiz, directorios, archivos in os.walk(ruta_test):
    for archivo in archivos:
        ruta_completa = os.path.join(raiz, archivo)
        try:
          #Obtención de características utilizando las funciones creadas
          feature_espectograma, sr = extraer_espectrograma(ruta_completa)
          feature_mfccs = obtener_caracteristicas_tradicionales(ruta_completa)
          #graficos de espectrogramas
          #graficar_espectrograma(feature_espectograma,sr)
        except:
          print('Error, no fue posible extraer caracteristicas del archivo',archivo)
          errores.append(archivo)
          with open(ruta_salida+'errores.txt', 'w', newline='', encoding='utf-8') as file:
            file.write(archivo)
        try:
          #guardando caracaterísticas en listas
          espectrograma.append(feature_espectograma)
          mfccs.append(feature_mfccs)
        except:
          print('Error, no fue posible guardar las caracteristicas en el listado',archivo)
          errores.append(archivo)
          with open(ruta_salida+'errores.txt', 'w', newline='', encoding='utf-8') as file:
            file.write(archivo)
        try:
          # generando tokens asociados a características
          mis_tokens_espectrograma, mi_codebook_espectrograma = tokenizar_con_vq(feature_espectograma)
          mis_tokens_mfccs, mi_codebook_mfccs = tokenizar_con_vq(feature_mfccs)

          # guardando tokens
          tokens_mfccs.append(mis_tokens_mfccs)
          tokens_espectrograma.append(mis_tokens_espectrograma)

          # guardando codebooks
          codebook_mfccs.append(mi_codebook_mfccs)
          codebook_espectrograma.append(mi_codebook_espectrograma)
        except:
          print('Error, no fue posible tokenizar las caracteristicas del archivo',archivo)
          errores.append(archivo)
          with open(ruta_salida+'errores.txt', 'w', newline='', encoding='utf-8') as file:
            file.write(archivo)

        np.save(ruta_salida+'espectrogramas/'+archivo[:-4]+'_espectrograma_entrenamiento.npy', feature_espectograma)
        np.save(ruta_salida+'mfccs/'+archivo[:-4]+'_mfccs_entrenamiento.npy', feature_mfccs)
        np.save(ruta_salida+'tokens_mfccs/'+archivo[:-4]+'_tokens_mfccs_entrenamiento.npy', mis_tokens_mfccs)
        np.save(ruta_salida+'tokens_espectrograma/'+archivo[:-4]+'_tokens_espectrograma_entrenamiento.npy', mis_tokens_espectrograma)
        np.save(ruta_salida+'codebooks/'+archivo[:-4]+'_codebook_mfccs_entrenamiento.npy', mi_codebook_mfccs)
        np.save(ruta_salida+'codebooks/'+archivo[:-4]+'_codebook_espectrograma_entrenamiento.npy', mi_codebook_espectrograma)

#**Cargar datos para entrenamiento**
cargar metadatos y diccionario

In [13]:
# Cargar metadatos y el diccionario (traductor de estilo a número)
df_train = pd.read_csv('train.csv')
estilos_dict = np.load('dict.npy', allow_pickle=True).item()

# Limpieza robusta: convertir a texto y eliminar espacios
df_train['label'] = df_train['label'].astype(str).str.strip()

# Crear la columna numérica usando el diccionario [cite: 42, 43]
df_train['label_idx'] = df_train['label'].map(estilos_dict)

# Verificar si quedaron etiquetas sin traducir (NaN)
faltantes = df_train[df_train['label_idx'].isna()]['label'].unique()
if len(faltantes) > 0:
    print(f"Estilos que no están en el diccionario: {faltantes}")

print(df_train)

        filename label     label_idx
0     081436.mp3     4  Instrumental
1     081457.mp3     4  Instrumental
2     081485.mp3     4  Instrumental
3     081491.mp3     4  Instrumental
4     081512.mp3     4  Instrumental
...          ...   ...           ...
6265  014735.mp3     7          Rock
6266  028274.mp3     7          Rock
6267  070409.mp3     7          Rock
6268  106955.mp3     7          Rock
6269  026681.mp3     7          Rock

[6270 rows x 3 columns]


In [51]:
def calculate_f1(y_true, y_pred_probs):
    """
    Calcula el F1-score macro.

    Parámetros:
    y_true:       etiquetas reales
    y_pred_probs: probabilidades predichas por el modelo (salida del softmax)

    Retorna:
    F1-score macro
    """
    # Convertir probabilidades a clases (argmax)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Calcular F1-score macro
    f1 = f1_score(y_true, y_pred, average='macro')

    return f1

In [ ]:

ruta_train = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Train/tokens/tokens_mfcc'

# 1. Cargar el CSV directamente (las etiquetas ya son números del 0 al 7)
df_train = pd.read_csv('train.csv')

# 2. Configurar la ruta donde están tus archivos individuales de tokens
CARPETA_TOKENS = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Train/tokens/tokens_mfcc'


tokens_filtrados = []
labels_filtradas = []
descartados_por_archivo = 0

print("Iniciando la carga y sincronización de archivos...")

# Recorremos el CSV usando los nombres correctos de tus columnas ('filename' y 'label')
for index, row in df_train.iterrows():
    nombre_base = row['filename'] # Tu columna se llama 'filename'
    etiqueta_numerica = row['label'] # Tu columna 'label' ya es un número (int64)
    print(index)

    # Asegurar que apunte al archivo .npy
    nombre_archivo_npy = f"{os.path.splitext(nombre_base)[0]}_mfccs_entrenamiento_tokens_mfccs_entrenamiento.npy"
    #nombre_archivo_npy = f"{os.path.splitext(nombre_base)[0]}.npy"
    ruta_completa = os.path.join(CARPETA_TOKENS, nombre_archivo_npy)

    # Verificar si el archivo de tokens realmente existe en el disco
    if not os.path.exists(ruta_completa):
        descartados_por_archivo += 1
        continue  # Si no existe, lo saltamos de forma segura

    # Cargamos el archivo de tokens individuales
    secuencia_tokens = np.load(ruta_completa)

    tokens_filtrados.append(secuencia_tokens)
    labels_filtradas.append(int(etiqueta_numerica)) # Guardamos el número directo

print('\n--- Resumen del Proceso ---')
print('Canciones procesadas y alineadas con éxito:',len(tokens_filtrados))
print('Archivos .npy no encontrados en la carpeta:',descartados_por_archivo)

# 3. Formato final con Padding para Keras
MAX_LEN = 500
X_final = pad_sequences(tokens_filtrados, maxlen=MAX_LEN, padding='post', truncating='post')

# Tienes 8 estilos musicales en total (del 0 al 7)
NUM_CLASES = 8
y_final = to_categorical(labels_filtradas, num_classes=NUM_CLASES)

print('\n--- Dimensiones Listas para Keras ---')
print(f"X_final: {X_final.shape}")  # Debería ser (Canciones_reales, 500)
print(f"y_final: {y_final.shape}")  # Debería ser (Canciones_reales, 8)

In [69]:
# Separar variables predictoras (X) y variable objetivo (y)
X = X_final
#y = y_final
y = np.argmax(y_final, axis=1)

# Paso 1: Separar el conjunto de prueba (10%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

# Paso 2: Separar entrenamiento (70%) y validación (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2/0.9, stratify=y_temp, random_state=42
)

# Verificar tamaños
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (4029, 500)
Validation: (1152, 500)
Test: (576, 500)


In [42]:
y.shape

(5757, 8)

##Configuración de hiperparámetros

In [79]:
# Espacios de búsqueda
activations_list  = ['relu', 'sigmoid', 'tanh']
depths            = [1, 2, 3]
neurons_list      = [20, 50, 100]
learning_rates    = [0.001, 0.01, 0.1]

# Parámetros fijos para la búsqueda inicial
fixed_batch_size  = 32
fixed_initializer = 'glorot_uniform'
fixed_optimizer   = 'SGD'
fixed_patience    = 50

# Valores para estudios de ablación
depths_study        = [1, 2, 3, 4, 5, 6]
neurons_study       = [10, 20, 50, 100, 150, 200]
learning_rates_study = np.logspace(-5, -1, 8)
batch_sizes_study   = [16, 32, 64, 128, 256]

#Modelamiento de la red profunda

In [86]:
def train_evaluate_model(activation, depth, neurons, learning_rate, optimizer_name='SGD',
                         batch_size=32, initializer='glorot_uniform', dropout_rate=0,
                         regularizer_type=None, regularizer_lambda=0, epochs=500, patience=50,
                         vocab_size=100, max_length=500):

    if regularizer_type == 'l1':
        reg = regularizers.l1(regularizer_lambda)
    elif regularizer_type == 'l2':
        reg = regularizers.l2(regularizer_lambda)
    else:
        reg = None

    model = Sequential()
    model.add(Input(shape=(max_length,), dtype='int32'))

    # Capa de Embedding + Reducción temporal para procesar tokens discretos
    model.add(Embedding(input_dim=vocab_size, output_dim=64))
    model.add(GlobalMaxPooling1D())

    # Primera capa oculta fully-connected usando tus hiperparámetros
    model.add(Dense(neurons,
                    activation=activation,
                    kernel_initializer=initializer,
                    kernel_regularizer=reg))

    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))

    # Capas ocultas adicionales según la profundidad
    for _ in range(depth - 1):
        model.add(Dense(neurons,
                        activation=activation,
                        kernel_initializer=initializer,
                        kernel_regularizer=reg))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    # Capa de salida para los 8 estilos musicales
    model.add(Dense(8, activation='softmax'))

    # Selección de optimizador
    if optimizer_name == 'SGD':
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == 'Adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'RMSprop':
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True
    )

    # Entrenamiento con tokens nativos
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    # Pasamos las probabilidades matrices (2D) directo a calculate_f1
    y_val_pred_probs = model.predict(X_val)
    y_test_pred_probs = model.predict(X_test)

    # Calculo de f1
    val_f1 = calculate_f1(y_val, y_val_pred_probs)
    test_f1 = calculate_f1(y_test, y_test_pred_probs)

    return history, val_f1, test_f1, model

In [87]:
import random

num_trials = 20
print(f"Realizando búsqueda con {num_trials} trials aleatorios")

results = []
best_f1 = -1
best_config = None

for i in range(num_trials):

    #  Elegir hiperparámetros aleatorios
    activation = random.choice(activations_list)
    depth = random.choice(depths)
    neurons = random.choice(neurons_list)
    lr = random.choice(learning_rates)

    print(f"\nTrial {i+1}/{num_trials}")
    print(f"Config: act={activation}, depth={depth}, neurons={neurons}, lr={lr}")

    #  Entrenar modelo
    history, val_f1, test_f1, model = train_evaluate_model(
        activation=activation,
        depth=depth,
        neurons=neurons,
        learning_rate=lr,
        optimizer_name=fixed_optimizer,
        batch_size=fixed_batch_size,
        initializer=fixed_initializer,
        epochs=200,
        patience=fixed_patience
    )

    #  Guardar resultados
    results.append({
        'activation': activation,
        'depth': depth,
        'neurons': neurons,
        'learning_rate': lr,
        'val_f1': val_f1,
        'test_f1': test_f1
    })

    #  Actualizar mejor modelo (según VALIDACIÓN)
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_config = results[-1]

        best_history = history
        best_model_keras = model

    print("Nuevo mejor modelo encontrado!")

#  Mostrar mejor configuración
print("\n===== MEJOR CONFIGURACIÓN =====")
print(best_config)

Realizando búsqueda con 20 trials aleatorios

Trial 1/20
Config: act=sigmoid, depth=2, neurons=20, lr=0.1
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Nuevo mejor modelo encontrado!

Trial 2/20
Config: act=tanh, depth=2, neurons=50, lr=0.1
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Nuevo mejor modelo encontrado!

Trial 3/20
Config: act=sigmoid, depth=2, neurons=100, lr=0.01
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Nuevo mejor modelo encontrado!

Trial 4/20
Config: act=tanh, depth=3, neurons=50, lr=0.001
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Nuevo mejor modelo encontrado!

Trial 5/20
Config: act=relu, depth=2, neurons=20, lr=0.1
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Nuevo mejor modelo encontrado!

Trial 6/20
Config: act=relu, depth=3, neurons=20, lr=0.001
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
18/18 ━━━